## Model From Scratch

In [1]:
# import pandas as pd

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [2]:
# import torch
# import torch.nn as nn

# class MCQTransformer(nn.Module):

#     def __init__(
#         self,
#         vocab_size,
#         hidden_dim=256,
#         max_length=256,
#         num_heads=8,
#         num_layers=4,
#         dropout=0.1
#     ):
#         super().__init__()

#         # Word embeddings
#         self.embedding = nn.Embedding(
#             vocab_size,
#             hidden_dim
#         )

#         # Position embeddings
#         self.position_embedding = nn.Embedding(
#             max_length,
#             hidden_dim
#         )

#         # One Transformer Encoder layer
#         encoder_layer = nn.TransformerEncoderLayer(
#             d_model=hidden_dim,
#             nhead=num_heads,
#             dim_feedforward=hidden_dim * 4,
#             dropout=dropout,
#             activation="gelu",
#             batch_first=True
#         )

#         # Stack multiple encoder layers
#         self.encoder = nn.TransformerEncoder(
#             encoder_layer,
#             num_layers=num_layers
#         )

#         self.dropout = nn.Dropout(dropout)

#         # Score one option
#         self.classifier = nn.Linear(
#             hidden_dim,
#             1
#         )

#     def forward(self, input_ids):

#         # input shape:
#         # (batch_size, 5, sequence_length)

#         B, C, L = input_ids.shape

#         # Convert into
#         # (batch_size*5, sequence_length)

#         input_ids = input_ids.view(B * C, L)

#         # Position ids
#         positions = torch.arange(
#             L,
#             device=input_ids.device
#         ).unsqueeze(0).expand(B * C, L)

#         # Embedding
#         x = self.embedding(input_ids)

#         # Add positional embedding
#         x = x + self.position_embedding(positions)

#         # Transformer
#         x = self.encoder(x)

#         # Mean Pooling
#         x = x.mean(dim=1)

#         x = self.dropout(x)

#         # Score
#         logits = self.classifier(x)

#         # Convert back to
#         # (batch_size,5)

#         logits = logits.view(B, C)

#         return logits

In [3]:
# from transformers import AutoTokenizer
# from torch.utils.data import Dataset, DataLoader

# # ---- Tokenizer ----
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# MAX_LEN = 256
# OPTION_COLS = ["A", "B", "C", "D", "E"]
# LABEL_MAP = {c: i for i, c in enumerate(OPTION_COLS)}


# class MCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_length=256, has_labels=True):
#         self.df = df.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.has_labels = has_labels

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         question = str(row["prompt"])

#         input_ids_per_option = []
#         for col in OPTION_COLS:
#             option_text = str(row[col])
            
#             enc = self.tokenizer(
#                 question,
#                 option_text,
#                 truncation=True,
#                 max_length=self.max_length,
#                 padding="max_length",
#                 return_tensors="pt"
#             )
#             input_ids_per_option.append(enc["input_ids"].squeeze(0))

#         # shape: (5, seq_len)
#         input_ids = torch.stack(input_ids_per_option, dim=0)

#         item = {"input_ids": input_ids}
#         if self.has_labels:
#             label = LABEL_MAP[str(row["answer"]).strip().upper()]
#             item["labels"] = torch.tensor(label, dtype=torch.long)
#         return item

# train_dataset = MCQDataset(train, tokenizer, max_length=MAX_LEN, has_labels=True)
# test_dataset = MCQDataset(test, tokenizer, max_length=MAX_LEN, has_labels=False)

# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [4]:
# device = torch.device(
#     "cuda" if torch.cuda.is_available() else "cpu"
# )

# model = MCQTransformer(
#     vocab_size=tokenizer.vocab_size,
#     hidden_dim=256,
#     max_length=256
# ).to(device)

# criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=3e-4,
#     weight_decay=0.01
# )

In [5]:
# epochs = 5

# for epoch in range(epochs):

#     model.train()

#     total_loss = 0

#     for batch in train_loader:

#         input_ids = batch["input_ids"].to(device)

#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()

#         logits = model(input_ids)

#         loss = criterion(
#             logits,
#             labels
#         )

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     print(
#         f"Epoch {epoch+1}:",
#         total_loss / len(train_loader)
#     )

In [6]:
# model.eval()

# all_logits = []

# with torch.no_grad():

#     for batch in test_loader:

#         input_ids = batch["input_ids"].to(device)

#         logits = model(input_ids)

#         all_logits.append(
#             logits.cpu()
#         )

# logits = torch.cat(all_logits).numpy()

In [7]:
# import numpy as np

# label_names = np.array(["A", "B", "C", "D", "E"])
# top3 = np.argsort(-logits, axis=1)[:, :3]

# predictions = [
#     " ".join(label_names[idx])
#     for idx in top3
# ]

# submission = test[["id"]].copy()
# submission["Prediction"] = predictions
# submission.to_csv("submission.csv", index=False)

## Deberta-V3

In [8]:
!pip install -q transformers datasets accelerate wandb

In [9]:
# import pandas as pd
# import numpy as np
# import torch
# import wandb
# from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
# from torch.utils.data import Dataset
# from sklearn.model_selection import train_test_split


# MODEL_NAME = "microsoft/deberta-v3-base"
# OPTIONS = ['A', 'B', 'C', 'D', 'E']
# MAP_LABEL = {opt: idx for idx, opt in enumerate(OPTIONS)}
# INV_MAP_LABEL = {idx: opt for idx, opt in enumerate(OPTIONS)}
# MAX_LENGTH = 256
# LABEL_COL = "answer"   
# QUESTION_COL = "prompt" 

# train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# def map3(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argsort(-logits, axis=1)
#     score = 0
#     for p, y in zip(preds, labels):
#         top3 = p[:3]
#         if y == top3[0]:
#             score += 1
#         elif y == top3[1]:
#             score += 0.5
#         elif y == top3[2]:
#             score += 1/3
#     return {"map3": score / len(labels)}


# class MCQDataset(Dataset):
#     def __init__(self, df, tokenizer, max_length=256, has_labels=True):
#         self.df = df.reset_index(drop=True)
#         self.tokenizer = tokenizer
#         self.max_length = max_length
#         self.has_labels = has_labels and (LABEL_COL in df.columns)

#     def __len__(self):
#         return len(self.df)

#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         question = str(row[QUESTION_COL])
#         choices = [str(row[opt]) for opt in OPTIONS]

#         encoding = self.tokenizer(
#             [question] * 5,
#             choices,
#             truncation=True,
#             padding="max_length",
#             max_length=self.max_length,
#             return_tensors="pt",
#         )

#         item = {
#             "input_ids": encoding["input_ids"],
#             "attention_mask": encoding["attention_mask"],
#         }

#         if self.has_labels:
#             label_val = row[LABEL_COL]
#             if isinstance(label_val, str):
#                 label_val = MAP_LABEL[label_val.strip().upper()]
#             item["labels"] = torch.tensor(int(label_val), dtype=torch.long)

#         return item

# train_df, valid_df = train_test_split(
#     train,
#     test_size=0.2,
#     random_state=42
# )
# train_df = train_df.reset_index(drop=True)
# valid_df = valid_df.reset_index(drop=True)

# train_dataset = MCQDataset(train_df, tokenizer, max_length=MAX_LENGTH, has_labels=True)
# valid_dataset = MCQDataset(valid_df, tokenizer, max_length=MAX_LENGTH, has_labels=True)


# wandb.login(key="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")
# wandb.init(
#     project='24f1002052-t22026',
#     name='Distilbert',
#     config={
#         'model':     MODEL_NAME,
#         'epochs':    5,
#         'lr':        2e-5,
#         'optimizer': 'adamw_torch',
#         'max_len':   256
#     }
# )


# model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

# training_args = TrainingArguments(
#     output_dir="./outputs",
#     learning_rate=2e-5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=15,
#     weight_decay=0.02,
#     warmup_ratio=0.1,
#     max_grad_norm=1.0,
#     logging_steps=50,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="map3",
#     greater_is_better=True,
#     optim="adafactor"
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=valid_dataset,
#     processing_class=tokenizer,
#     compute_metrics=map3
# )

# trainer.train()
# wandb.finish()


# test_dataset = MCQDataset(test, tokenizer, max_length=MAX_LENGTH, has_labels=False)
# pred = trainer.predict(test_dataset)
# logits = pred.predictions

# labels_arr = np.array(OPTIONS)
# order = np.argsort(-logits, axis=1)
# predictions = [
#     " ".join(labels_arr[idx[:3]])
#     for idx in order
# ]

# submission = pd.DataFrame({
#     "id": test["id"],
#     "Prediction": predictions
# })
# submission.to_csv("submission.csv", index=False)
# print("Saved submission.csv")
# print(submission.head())

## DistilBert Model

In [10]:
!pip install -q transformers datasets accelerate wandb

In [11]:
import torch
import wandb
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

In [12]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [13]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForMultipleChoice LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from torch.utils.data import Dataset
import torch

class MCQDataset(Dataset):

    def __init__(self, df, tokenizer, max_length=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        question = row["prompt"]

        choices = [
            row["A"],
            row["B"],
            row["C"],
            row["D"],
            row["E"],
        ]

        encoding = self.tokenizer(
            [question] * 5,
            choices,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
        }

        # Only add labels for train/validation
        if "label" in self.df.columns:
            item["labels"] = torch.tensor(row["label"], dtype=torch.long)

        return item

In [15]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

train["label"] = encoder.fit_transform(train["answer"])

In [16]:
import numpy as np

def map3(eval_pred):

    logits, labels = eval_pred

    preds = np.argsort(-logits, axis=1)

    score = 0

    for p, y in zip(preds, labels):

        top3 = p[:3]

        if y == top3[0]:
            score += 1

        elif y == top3[1]:
            score += 0.5

        elif y == top3[2]:
            score += 1/3

    return {"map3": score / len(labels)}

In [17]:
from sklearn.model_selection import train_test_split

train_df, valid_df = train_test_split(
    train,
    test_size=0.2,
    stratify=train["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

train_dataset = MCQDataset(
    train_df,
    tokenizer,
    max_length=256
)

valid_dataset = MCQDataset(
    valid_df,
    tokenizer,
    max_length=256
)

In [18]:
wandb.login(key="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")
wandb.init(
    project='24f1002052-t22026',
    name='Distilbert',
    config={
        'model':     MODEL_NAME,
        'epochs':    5,
        'lr':        2e-5,
        'optimizer': 'adamw_torch',
        'max_len':   256
    }
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [19]:

training_args = TrainingArguments(

    output_dir="./outputs",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=15,

    weight_decay=0.02,

    warmup_ratio=0.1,

    max_grad_norm=1.0,

    logging_steps=50,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="map3",

    greater_is_better=True,

    optim="adamw_torch"
)



warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    compute_metrics=map3
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,3.186882,3.118552,0.699167
2,1.946691,1.435663,0.915833
3,0.865279,0.587847,0.980000
4,0.420832,0.306371,0.990000
5,0.193464,0.193359,0.995000
6,0.115734,0.132856,0.995000
7,0.045072,0.061688,0.998750
8,0.041730,0.062904,1.000000
9,0.025396,0.031532,1.000000
10,0.017286,0.009047,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1500, training_loss=0.5103395426372687, metrics={'train_runtime': 1606.9354, 'train_samples_per_second': 14.935, 'train_steps_per_second': 0.933, 'total_flos': 7947902177280000.0, 'train_loss': 0.5103395426372687, 'epoch': 15.0})

In [21]:
test_dataset = MCQDataset(
    test,
    tokenizer,
    max_length=256
)

In [22]:
pred = trainer.predict(test_dataset)

logits = pred.predictions

labels = np.array(["A", "B", "C", "D", "E"])

order = np.argsort(-logits, axis=1)

predictions = [
    " ".join(labels[idx[:3]])
    for idx in order
]

wandb.finish()

In [23]:
print(len(pred.predictions))

500


In [24]:
submission = pd.DataFrame({
    "id": test["id"],
    "Prediction": predictions
})

submission.to_csv("submission.csv", index=False)

## RAG

In [25]:
# !pip install pymupdf faiss-cpu wandb datasets -q

In [26]:
# import fitz   # pymupdf
# import pandas as pd
# import numpy as np
# import re
# import os

# def extract_pdf_chunks(pdf_path, chunk_size=80, overlap=20):
    
#     doc   = fitz.open(pdf_path)
#     pages = []

#     for page_num in range(len(doc)):
#         page = doc[page_num]
#         text = page.get_text()

#         text = re.sub(r'\s+', ' ', text)          
#         text = re.sub(r'[^\w\s\.\,\;\:\-\(\)]', ' ', text)  # remove special chars
#         text = text.strip()

#         if len(text) > 50:   # skip very short pages
#             pages.append({
#                 'page': page_num + 1,
#                 'text': text
#             })

#     doc.close()

#     full_text = ' '.join([p['text'] for p in pages])
#     words = full_text.split()
    
#     chunks = []
#     i = 0
#     while i < len(words):
#         chunk_words = words[i:i + chunk_size]
#         chunk_text  = ' '.join(chunk_words)

#         if len(chunk_text) > 50:   # skip tiny chunks
#             chunks.append(chunk_text)

#         i += (chunk_size - overlap)   # overlap for context continuity

#     return chunks


# PDF_DIR = '/kaggle/input/datasets/aadity7531/rag-dataset'

# pdfs = {
#     'Physics_11_part1':   'NCERT-Class-11-Physics-Part-1.pdf',
#     'Physics_11_part2':   'NCERT-Class-11-Physics-Part-2.pdf',
#     'Physics_12_part1':   'NCERT-Class-12-Physics-Part-1.pdf',
#     'Physics_12_part2':   'NCERT-Class-12-Physics-Part-2.pdf',
#     'Chemistry_11_part1': 'NCERT-Class-11-Chemistry-Part-1.pdf',
#     'Chemistry_11_part2': 'NCERT-Class-11-Chemistry-Part-2.pdf',
#     'Chemistry_12_part1': 'NCERT-Class-12-Chemistry-Part-1.pdf',
#     'Chemistry_12_part2': 'NCERT-Class-12-Chemistry-Part-2.pdf',
#     'geography1': 'Fundamental of Physical Geography (Class XI) 2.pdf',
#     'geography2': 'India Physical Environment (Class XI) 2.pdf',
#     'geography3': '/kaggle/input/datasets/aadity7531/rag-dataset/Practical Work in Geography Part 1.pdf',
#     'class9': '/kaggle/input/datasets/aadity7531/rag-dataset/class 9 abc.pdf',
#     'history1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-10-History.pdf',
#     'history2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-History.pdf',
#     'Biology_11': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-11-Biology.pdf',
#     'Biology_12': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-Biology.pdf',
#     'history_12_1': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-1 (1).pdf',
#     'History_12_2': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-2.pdf',
#     'History_12_3': '/kaggle/input/datasets/aadity7531/rag-dataset/NCERT-Class-12-History-Part-3.pdf',
    
# }

# all_chunks = []

# for subject, filename in pdfs.items():
#     path = os.path.join(PDF_DIR, filename)
    
#     chunks = extract_pdf_chunks(path, chunk_size=100, overlap=30)

#     for chunk in chunks:
#         all_chunks.append({
#             'subject': subject,
#             'text':    chunk
#         })

#     print(f"{subject}: {len(chunks)} chunks extracted")

# knowledge_df = pd.DataFrame(all_chunks)
# print(knowledge_df.iloc[10]['text'][:300])

In [27]:
# import faiss
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.preprocessing import normalize

# docs = knowledge_df['text'].tolist()

# # TF-IDF vectorizer on NCERT text
# tfidf = TfidfVectorizer(
#     max_features=15000,
#     ngram_range=(1, 2),
#     min_df=2,
#     strip_accents='unicode'
# )

# doc_vecs = tfidf.fit_transform(docs).toarray().astype(np.float32)
# doc_vecs  = normalize(doc_vecs, norm='l2')

# # FAISS index
# DIM   = doc_vecs.shape[1]
# index = faiss.IndexFlatIP(DIM)
# index.add(doc_vecs)

# print(f"FAISS ready — {index.ntotal} knowledge chunks, dim={DIM}")

In [28]:
# def get_ncert_context(prompt: str, options: dict, top_k: int = 3) -> str:

#     query = prompt + ' ' + ' '.join(options.values())
#     query = query[:500]   # limit query length

#     vec = tfidf.transform([query]).toarray().astype(np.float32)
#     vec = normalize(vec, norm='l2')

#     scores, idxs = index.search(vec, top_k + 2)

#     parts = []
#     for score, idx in zip(scores[0], idxs[0]):
#         if idx < 0:
#             continue
#         if score < 0.15:       # skip irrelevant chunks
#             continue
#         if len(parts) >= top_k:
#             break

#         chunk = docs[int(idx)][:250]   # truncate for token budget
#         parts.append(chunk)

#     return ' || '.join(parts) if parts else 'No relevant context.'



# test_q = "What is the relationship between Hamiltonians in quantum mechanics?"
# test_opts = {'A': 'same energy', 'B': 'higher energy', 'C': 'different spin', 'D': 'different energy', 'E': 'lower energy'}

# ctx = get_ncert_context(test_q, test_opts)
# print("Query:", test_q)
# print("\nRetrieved NCERT context:")
# print(ctx[:400])

In [29]:
# import pandas as pd

# train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# OPTIONS  = ['A', 'B', 'C', 'D', 'E']
# LABEL2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
# ID2LABEL = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

# for col in OPTIONS + ['prompt']:
#     train[col] = train[col].fillna('none')
#     test[col]  = test[col].fillna('none')

# train['label'] = train['answer'].map(LABEL2ID)
# train = train.reset_index(drop=True)
# test  = test.reset_index(drop=True)

# # Build contexts
# print("Building NCERT RAG context for train...")
# train_ctx = []
# for i in range(len(train)):
#     opts = {opt: train.iloc[i][opt] for opt in OPTIONS}
#     ctx  = get_ncert_context(train.iloc[i]['prompt'], opts)
#     train_ctx.append(ctx)
#     if i % 400 == 0:
#         print(f"  {i}/2000")

# train['rag_context'] = train_ctx

# test_ctx = []
# for i in range(len(test)):
#     opts = {opt: test.iloc[i][opt] for opt in OPTIONS}
#     ctx  = get_ncert_context(test.iloc[i]['prompt'], opts)
#     test_ctx.append(ctx)
#     if i % 100 == 0:
#         print(f"  {i}/500")

# test['rag_context'] = test_ctx

# real_ctx = (train['rag_context'] != 'No relevant context.').sum()
# print(f"Questions with real NCERT context: {real_ctx}/2000 ({real_ctx/20:.1f}%)")

In [30]:
# import torch
# from transformers import AutoTokenizer
# from datasets import Dataset

# DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# MODEL_NAME = '/kaggle/input/models/mtnash/deberta-v3-small/transformers/default/1'
# MAX_LEN    = 256
# EPOCHS     = 2
# LR         = 1e-5
# BATCH_SIZE = 4

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# def tokenize_fn(examples):
#     all_input_ids      = []
#     all_attention_mask = []

#     for i in range(len(examples['prompt'])):
        
#         ctx    = str(examples['rag_context'][i])[:150]
#         prompt = str(examples['prompt'][i])

#         # Format: "Context: <ncert_text> Question: <prompt>"
#         first_seq = f"Context: {ctx} Question: {prompt}"

#         opt_ids = []
#         opt_att = []

#         for opt in OPTIONS:
#             second_seq = str(examples[opt][i])

#             enc = tokenizer(
#                 first_seq,
#                 second_seq,
#                 max_length=MAX_LEN,
#                 truncation=True,
#                 padding='max_length',
#                 return_tensors=None,
#             )
#             opt_ids.append([int(x) for x in enc['input_ids']])
#             opt_att.append([int(x) for x in enc['attention_mask']])

#         all_input_ids.append(opt_ids)
#         all_attention_mask.append(opt_att)

#     return {
#         'input_ids':      all_input_ids,
#         'attention_mask': all_attention_mask,
#     }

In [31]:
# from sklearn.model_selection import train_test_split

# KEEP     = OPTIONS + ['prompt', 'rag_context']
# KEEP_LBL = KEEP + ['label']

# # Use full 2000 for training
# train_hf = Dataset.from_pandas(train[KEEP_LBL].copy())
# test_hf  = Dataset.from_pandas(test[KEEP].copy())


# train_tok = train_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

# test_tok  = test_hf.map(tokenize_fn, batched=True, batch_size=32, remove_columns=KEEP)

# FEAT = ['input_ids', 'attention_mask']
# train_tok.set_format(type='torch', columns=FEAT + ['label'])
# test_tok.set_format( type='torch', columns=FEAT)

# print("Done!")
# print("Shape:", train_tok[0]['input_ids'].shape)  # (5, 256)

In [32]:
# import gc
# import wandb
# import torch
# import torch.nn as nn
# import pandas as pd
# from torch.optim import AdamW
# from torch.utils.data import DataLoader
# from transformers import AutoModelForMultipleChoice, get_cosine_schedule_with_warmup
# from sklearn.metrics import f1_score, accuracy_score

# def collate_train(features):
#     return {
#         'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
#         'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
#         'labels':         torch.stack([f['label']          for f in features]).long(),
#     }

# def collate_test(features):
#     return {
#         'input_ids':      torch.stack([f['input_ids']      for f in features]).long(),
#         'attention_mask': torch.stack([f['attention_mask'] for f in features]).long(),
#     }


# train_loader = DataLoader(
#     train_tok, batch_size=BATCH_SIZE,
#     shuffle=True,  collate_fn=collate_train
# )
# test_loader = DataLoader(
#     test_tok, batch_size=BATCH_SIZE,
#     shuffle=False, collate_fn=collate_test,
# )


# wandb.login(key="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")
# wandb.init(
#     project='24f1002052-t22026',
#     name='deberta-NCERT-RAG',
#     config={
#         'model':     MODEL_NAME,
#         'knowledge': 'NCERT Physics+Chemistry 11+12',
#         'epochs':    EPOCHS,
#         'lr':        LR,
#         'max_len':   MAX_LEN
#     }
# )


# criterion = nn.CrossEntropyLoss()

# model     = AutoModelForMultipleChoice.from_pretrained(
#     MODEL_NAME, ignore_mismatched_sizes=True
# ).to(DEVICE)

# optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# total_steps = len(train_loader) * EPOCHS

# scheduler   = get_cosine_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=total_steps // 10,
#     num_training_steps=total_steps,
# )

# best_loss  = float('inf')
# best_state = None

# for epoch in range(EPOCHS):
#     model.train()
#     total_loss  = 0.0
#     valid_steps = 0

#     for step, batch in enumerate(train_loader):
#         batch  = {k: v.to(DEVICE) for k, v in batch.items()}
#         labels = batch.pop('labels')
#         optimizer.zero_grad()
        
#         out  = model(**batch)
#         loss = criterion(out.logits, labels)

#         loss.backward()
#         torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
#         optimizer.step()
#         scheduler.step()

#         total_loss  += loss.item()
#         valid_steps += 1

#         if step % 50 == 0:
#             print(f"  Step {step}/{len(train_loader)} loss={loss.item():.4f}")

#     avg = total_loss / max(valid_steps, 1)
#     wandb.log({'epoch': epoch+1, 'train_loss': avg})

#     if avg < best_loss:
#         best_loss  = avg
#         best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
#         print(f"  ✅ Best saved! Loss={avg:.4f}")

# wandb.finish()
# print("Training done!")


# model.load_state_dict(best_state)
# model.to(DEVICE)
# model.eval()

# all_preds = []
# with torch.no_grad():
#     for batch in test_loader:
#         batch = {k: v.to(DEVICE) for k, v in batch.items()}
#         out   = model(**batch)
#         preds = torch.argmax(out.logits, dim=1)
#         all_preds.extend(preds.cpu().numpy())

# pred_labels = [ID2LABEL[int(p)] for p in all_preds]

# sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
# sub['Prediction'] = pred_labels
# sub.to_csv('/kaggle/working/submission.csv', index=False)
# print("✅ Submission saved!")
# print(sub.head())